# Movie & OTT Analytics — Python Mini Project

**Goal:** Analyze a movie dataset using Python, Pandas, NumPy, Matplotlib and Seaborn, then export cleaned/aggregated tables that can be imported into **Power BI** and **Tableau**.

### Questions answered
1. How are movies distributed across OTT platforms?
2. Which genres have the highest ratings?
3. Which directors have the strongest average ratings?
4. How does runtime relate to IMDb rating?
5. How does box office compare with budget?
6. Which countries/languages dominate the dataset?
7. How have movie releases changed over time?
8. Which platforms have the highest average rating?
9. Which movies have the strongest commercial performance?
10. What tables should be exported for Power BI/Tableau?

> The included CSV is a **synthetic sample dataset for academic/demo use**, not official movie/OTT data.


In [ ]:
# 1. Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

sns.set_theme(style="whitegrid")

DATA_FILE = Path("movie_ott_dataset.csv")


In [ ]:
# 2. Load the dataset
df = pd.read_csv(DATA_FILE)

print("Shape:", df.shape)
display(df.head())


## 3. Initial data inspection

In [ ]:
# Basic structure and quality checks
print("Data types:")
display(df.dtypes.to_frame("dtype"))

print("\nMissing values:")
display(df.isna().sum().to_frame("missing"))

print("\nDuplicate rows:", df.duplicated().sum())

print("\nSummary statistics:")
display(df.describe(include="all").T)


## 4. Data cleaning and feature engineering

In [ ]:
# Clean text columns
text_cols = ["title", "genre", "platform", "director", "country", "language"]
for col in text_cols:
    df[col] = df[col].astype(str).str.strip()

# Convert release_year from the dataset's "theatrical_year,ott_year" format
df[["theatrical_year", "ott_year"]] = df["release_year"].str.split(",", expand=True).astype(int)

# Useful calculated metrics
df["profit_musd"] = df["box_office_musd"] - df["budget_musd"]
df["roi_percent"] = np.where(
    df["budget_musd"] > 0,
    (df["profit_musd"] / df["budget_musd"]) * 100,
    np.nan
)

df["rating_band"] = pd.cut(
    df["imdb_rating"],
    bins=[0, 6.9, 7.9, 8.5, 10],
    labels=["Below 7", "7–7.9", "8–8.5", "Above 8.5"]
)

df["runtime_category"] = pd.cut(
    df["runtime_min"],
    bins=[0, 120, 150, 180, np.inf],
    labels=["Short (≤120)", "Medium (121–150)", "Long (151–180)", "Very Long (>180)"]
)

df["commercial_success"] = pd.cut(
    df["roi_percent"],
    bins=[-np.inf, 0, 100, 300, np.inf],
    labels=["Loss", "Low ROI", "Good ROI", "High ROI"]
)

display(df.head())


## 5. KPI summary

In [ ]:
# High-level KPIs
kpis = {
    "Total Movies": len(df),
    "Average IMDb Rating": df["imdb_rating"].mean(),
    "Average Runtime (min)": df["runtime_min"].mean(),
    "Total Box Office ($M)": df["box_office_musd"].sum(),
    "Total Budget ($M)": df["budget_musd"].sum(),
    "Total Profit ($M)": df["profit_musd"].sum(),
    "Average ROI (%)": df["roi_percent"].mean(),
    "OTT Platforms": df["platform"].nunique(),
    "Genres": df["genre"].nunique(),
    "Countries": df["country"].nunique()
}

kpi_df = pd.DataFrame(kpis.items(), columns=["Metric", "Value"])
display(kpi_df)


## 6. Exploratory Data Analysis

In [ ]:
# 6.1 Movies by OTT platform
platform_counts = df["platform"].value_counts()

plt.figure(figsize=(10, 5))
sns.barplot(x=platform_counts.values, y=platform_counts.index)
plt.title("Number of Movies by OTT Platform")
plt.xlabel("Number of Movies")
plt.ylabel("Platform")
plt.tight_layout()
plt.show()


In [ ]:
# 6.2 Average IMDb rating by genre
genre_rating = (
    df.groupby("genre", as_index=False)["imdb_rating"]
      .mean()
      .sort_values("imdb_rating", ascending=False)
)

plt.figure(figsize=(10, 5))
sns.barplot(data=genre_rating, x="imdb_rating", y="genre")
plt.title("Average IMDb Rating by Genre")
plt.xlabel("Average IMDb Rating")
plt.ylabel("Genre")
plt.xlim(0, 10)
plt.tight_layout()
plt.show()

display(genre_rating)


In [ ]:
# 6.3 Movie releases by theatrical year
year_counts = df["theatrical_year"].value_counts().sort_index()

plt.figure(figsize=(10, 5))
sns.lineplot(x=year_counts.index, y=year_counts.values, marker="o")
plt.title("Movies by Theatrical Release Year")
plt.xlabel("Year")
plt.ylabel("Number of Movies")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# 6.4 Runtime vs IMDb rating
plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=df,
    x="runtime_min",
    y="imdb_rating",
    hue="genre",
    s=100
)
plt.title("Runtime vs IMDb Rating")
plt.xlabel("Runtime (minutes)")
plt.ylabel("IMDb Rating")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

print("Correlation between runtime and rating:",
      round(df["runtime_min"].corr(df["imdb_rating"]), 3))


In [ ]:
# 6.5 Budget vs Box Office
plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=df,
    x="budget_musd",
    y="box_office_musd",
    hue="genre",
    s=100
)
plt.title("Budget vs Box Office")
plt.xlabel("Budget ($M)")
plt.ylabel("Box Office ($M)")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

print("Correlation between budget and box office:",
      round(df["budget_musd"].corr(df["box_office_musd"]), 3))


In [ ]:
# 6.6 Movies by country
country_counts = df["country"].value_counts()

plt.figure(figsize=(8, 5))
sns.barplot(x=country_counts.values, y=country_counts.index)
plt.title("Movies by Country")
plt.xlabel("Number of Movies")
plt.ylabel("Country")
plt.tight_layout()
plt.show()


In [ ]:
# 6.7 Language distribution
language_counts = df["language"].value_counts()

plt.figure(figsize=(8, 5))
plt.pie(language_counts.values, labels=language_counts.index, autopct="%1.1f%%", startangle=90)
plt.title("Movie Language Distribution")
plt.tight_layout()
plt.show()


## 7. Platform analysis

In [ ]:
platform_analysis = (
    df.groupby("platform")
      .agg(
          movie_count=("movie_id", "count"),
          avg_rating=("imdb_rating", "mean"),
          avg_runtime=("runtime_min", "mean"),
          total_box_office_musd=("box_office_musd", "sum"),
          avg_roi_percent=("roi_percent", "mean")
      )
      .sort_values("avg_rating", ascending=False)
      .reset_index()
)

display(platform_analysis)


In [ ]:
# Platform rating comparison
plt.figure(figsize=(10, 5))
sns.barplot(data=platform_analysis, x="avg_rating", y="platform")
plt.title("Average IMDb Rating by OTT Platform")
plt.xlabel("Average IMDb Rating")
plt.ylabel("Platform")
plt.xlim(0, 10)
plt.tight_layout()
plt.show()


## 8. Genre analysis

In [ ]:
genre_analysis = (
    df.groupby("genre")
      .agg(
          movie_count=("movie_id", "count"),
          avg_rating=("imdb_rating", "mean"),
          avg_runtime=("runtime_min", "mean"),
          total_box_office_musd=("box_office_musd", "sum"),
          avg_roi_percent=("roi_percent", "mean")
      )
      .sort_values("avg_rating", ascending=False)
      .reset_index()
)

display(genre_analysis)


## 9. Director analysis

In [ ]:
director_analysis = (
    df.groupby("director")
      .agg(
          movie_count=("movie_id", "count"),
          avg_rating=("imdb_rating", "mean"),
          total_box_office_musd=("box_office_musd", "sum"),
          avg_roi_percent=("roi_percent", "mean")
      )
      .query("movie_count >= 2")
      .sort_values(["avg_rating", "total_box_office_musd"], ascending=False)
      .reset_index()
)

display(director_analysis)


## 10. Top movies

In [ ]:
# Top-rated movies
top_rated = (
    df.sort_values(["imdb_rating", "box_office_musd"], ascending=[False, False])
      [["title", "genre", "platform", "imdb_rating", "director"]]
      .head(10)
)

print("Top-rated movies")
display(top_rated)

# Highest box-office movies
top_box_office = (
    df.sort_values("box_office_musd", ascending=False)
      [["title", "genre", "platform", "box_office_musd", "budget_musd", "profit_musd"]]
      .head(10)
)

print("Highest box-office movies")
display(top_box_office)

# Highest ROI movies
top_roi = (
    df[df["budget_musd"] > 0]
      .sort_values("roi_percent", ascending=False)
      [["title", "genre", "platform", "budget_musd", "box_office_musd", "roi_percent"]]
      .head(10)
)

print("Highest ROI movies")
display(top_roi)


## 11. Correlation analysis

In [ ]:
numeric_cols = [
    "runtime_min", "imdb_rating", "box_office_musd",
    "budget_musd", "profit_musd", "roi_percent"
]

corr = df[numeric_cols].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

display(corr)


## 12. OTT availability analysis

In [ ]:
# How many movies from each genre appear on each platform?
platform_genre = pd.crosstab(df["platform"], df["genre"])

plt.figure(figsize=(12, 6))
sns.heatmap(platform_genre, annot=True, fmt="d", cmap="Blues")
plt.title("Genre Availability Across OTT Platforms")
plt.xlabel("Genre")
plt.ylabel("OTT Platform")
plt.tight_layout()
plt.show()

display(platform_genre)


## 13. Create a dashboard-ready master table

In [ ]:
# Final table for BI tools
dashboard_data = df.copy()

# Reorder columns for readability
dashboard_columns = [
    "movie_id", "title", "genre", "platform", "theatrical_year", "ott_year",
    "director", "runtime_min", "imdb_rating", "budget_musd",
    "box_office_musd", "profit_musd", "roi_percent", "rating_band",
    "runtime_category", "commercial_success", "country", "language"
]

dashboard_data = dashboard_data[dashboard_columns]

display(dashboard_data.head())


## 14. Export files for Power BI and Tableau

In [ ]:
# Export dashboard-ready tables.
dashboard_data.to_csv("movie_ott_dashboard_data.csv", index=False)
platform_analysis.to_csv("ott_platform_analysis.csv", index=False)
genre_analysis.to_csv("ott_genre_analysis.csv", index=False)
director_analysis.to_csv("director_analysis.csv", index=False)
top_rated.to_csv("top_rated_movies.csv", index=False)
top_box_office.to_csv("top_box_office_movies.csv", index=False)
top_roi.to_csv("top_roi_movies.csv", index=False)

print("Export completed.")


## 15. Suggested Power BI dashboard

Import **`movie_ott_dashboard_data.csv`** into Power BI.

### Page 1 — Executive Overview
- Card: Total Movies
- Card: Average IMDb Rating
- Card: Total Box Office
- Card: Average ROI
- Bar chart: Movies by OTT Platform
- Bar chart: Movies by Genre
- Line chart: Movies by Release Year

### Page 2 — OTT Platform Analysis
- Platform slicer
- Average rating by platform
- Movie count by platform
- Total box office by platform
- Genre availability heatmap/matrix

### Page 3 — Movie Performance
- Top 10 movies by rating
- Top 10 by box office
- Top 10 by ROI
- Budget vs box-office scatter plot

### Useful slicers
- Platform
- Genre
- Country
- Language
- Theatrical year
- Rating band


## 16. Suggested Tableau dashboard

Use **`movie_ott_dashboard_data.csv`**.

Create a Tableau story with:

**Sheet 1 — OTT Landscape**
- Platform-wise movie count
- Platform rating comparison

**Sheet 2 — Genre Trends**
- Genre rating
- Genre movie count
- Genre box office

**Sheet 3 — Commercial Performance**
- Budget vs box office
- ROI by movie
- Profit distribution

**Sheet 4 — Movie Explorer**
- Filters for platform, genre, language and year
- Movie-level table
- Rating and box-office views


## 17. Key findings generated from the sample data

Run the following cell to automatically produce concise findings instead of manually writing them.


In [ ]:
# Automatically generate findings for the project report
best_rated_movie = df.loc[df["imdb_rating"].idxmax()]
highest_box_office = df.loc[df["box_office_musd"].idxmax()]
highest_roi = df.loc[df["roi_percent"].idxmax()]

best_platform = platform_analysis.sort_values("avg_rating", ascending=False).iloc[0]
best_genre = genre_analysis.sort_values("avg_rating", ascending=False).iloc[0]

findings = [
    f"Highest-rated movie: {best_rated_movie['title']} ({best_rated_movie['imdb_rating']}/10).",
    f"Highest box-office movie: {highest_box_office['title']} (${highest_box_office['box_office_musd']:.0f}M).",
    f"Highest ROI movie: {highest_roi['title']} ({highest_roi['roi_percent']:.1f}% ROI).",
    f"Platform with the highest average rating in this sample: {best_platform['platform']} ({best_platform['avg_rating']:.2f}).",
    f"Highest-rated genre in this sample: {best_genre['genre']} ({best_genre['avg_rating']:.2f}).",
    f"Budget vs box-office correlation: {df['budget_musd'].corr(df['box_office_musd']):.2f}.",
    f"Runtime vs IMDb rating correlation: {df['runtime_min'].corr(df['imdb_rating']):.2f}."
]

for i, finding in enumerate(findings, 1):
    print(f"{i}. {finding}")


## 18. Conclusion

This project demonstrates a complete data-visualization workflow:

**Raw movie/OTT data → Data cleaning → Feature engineering → EDA → Statistical analysis → Visualization → BI-ready exports → Power BI/Tableau dashboards**

For an academic submission, clearly state that the included dataset is synthetic/sample data unless you replace it with a real, properly sourced dataset.
